# Day 093 | Ground Motion Selection using Clustering

## 100 Days of ML for Geotechnical Engineering
**Phase:** Geotechnical AI Projects

**Author:** Ripon Chandra Malo

---

## Project Overview

**Objective:** Use K-Means and hierarchical clustering to select representative ground motions from a library of 100 synthetic earthquake records. In seismic geotechnical analysis, engineers need a small set of representative motions for site response or liquefaction triggering studies. Clustering groups similar records and lets us pick one per cluster.

**Workflow:**
1. Generate 100 synthetic ground motion records with features (PGA, PGV, Arias Intensity, duration, spectral acceleration)
2. Standardize features
3. Apply K-Means clustering
4. Apply hierarchical clustering with dendrogram
5. Select one representative motion per cluster
6. Compare the two approaches

---

## Part 1: Imports and Setup

In [ ]:
# ─────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded.")

---

## Part 2: Generate Synthetic Ground Motion Library

We create 100 records with realistic intensity measures (IMs). The records come from three synthetic source types that produce different characteristic signatures.

In [ ]:
# ─────────────────────────────────────────────────────
# GENERATE 100 SYNTHETIC GROUND MOTION RECORDS
# ─────────────────────────────────────────────────────

n_records = 100

# Three source types: shallow crustal, subduction, near-fault
source_type = np.random.choice(['Crustal', 'Subduction', 'Near-Fault'], 
                                size=n_records, p=[0.5, 0.3, 0.2])

records = []
for i in range(n_records):
    src = source_type[i]
    if src == 'Crustal':
        pga = np.random.uniform(0.05, 0.40)
        pgv = pga * np.random.uniform(60, 100)
        duration = np.random.uniform(10, 30)
        arias = pga**2 * duration * np.random.uniform(0.3, 0.8)
    elif src == 'Subduction':
        pga = np.random.uniform(0.03, 0.25)
        pgv = pga * np.random.uniform(80, 140)
        duration = np.random.uniform(30, 90)
        arias = pga**2 * duration * np.random.uniform(0.4, 1.0)
    else:  # Near-Fault
        pga = np.random.uniform(0.20, 0.70)
        pgv = pga * np.random.uniform(100, 180)
        duration = np.random.uniform(5, 20)
        arias = pga**2 * duration * np.random.uniform(0.5, 1.2)
    
    # Spectral accelerations at key periods
    sa_02 = pga * np.random.uniform(1.5, 3.0)   # Sa(0.2s)
    sa_10 = pga * np.random.uniform(0.4, 1.2)   # Sa(1.0s)
    sa_20 = pga * np.random.uniform(0.1, 0.5)   # Sa(2.0s)
    
    records.append({
        'Record_ID': f'GM-{i+1:03d}',
        'Source': src,
        'PGA_g': round(pga, 4),
        'PGV_cm_s': round(pgv, 2),
        'Arias_m_s': round(arias, 4),
        'Duration_s': round(duration, 1),
        'Sa_02_g': round(sa_02, 4),
        'Sa_10_g': round(sa_10, 4),
        'Sa_20_g': round(sa_20, 4),
    })

df = pd.DataFrame(records)
print(f"Generated {len(df)} ground motion records")
print(f"\nSource distribution:")
print(df['Source'].value_counts())
print(f"\nFirst 5 records:")
df.head()

In [ ]:
# ─────────────────────────────────────────────────────
# SUMMARY STATISTICS
# ─────────────────────────────────────────────────────

feature_cols = ['PGA_g', 'PGV_cm_s', 'Arias_m_s', 'Duration_s', 
                'Sa_02_g', 'Sa_10_g', 'Sa_20_g']

print("Feature statistics:")
df[feature_cols].describe().round(4)

---

## Part 3: Feature Exploration and Standardization

In [ ]:
# ─────────────────────────────────────────────────────
# VISUALIZE FEATURE DISTRIBUTIONS BY SOURCE TYPE
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

colors = {'Crustal': '#2196F3', 'Subduction': '#FF9800', 'Near-Fault': '#F44336'}

for i, col in enumerate(feature_cols):
    ax = axes[i]
    for src, color in colors.items():
        mask = df['Source'] == src
        ax.hist(df.loc[mask, col], bins=15, alpha=0.5, 
                label=src, color=color, edgecolor='white')
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.set_ylabel('Count')
    if i == 0:
        ax.legend(fontsize=8)

axes[-1].axis('off')
fig.suptitle('Ground Motion Feature Distributions by Source Type', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────
# CORRELATION MATRIX
# ─────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(8, 6))
corr = df[feature_cols].corr()
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(feature_cols)))
ax.set_yticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(feature_cols, fontsize=9)
for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax, label='Correlation')
ax.set_title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────
# STANDARDIZE FEATURES
# ─────────────────────────────────────────────────────

scaler = StandardScaler()
X = scaler.fit_transform(df[feature_cols])

print(f"Scaled feature matrix shape: {X.shape}")
print(f"Mean after scaling: {X.mean(axis=0).round(6)}")
print(f"Std after scaling:  {X.std(axis=0).round(6)}")
print("\nStandardization ensures PGA (0-1 g) and Duration (5-90 s)")
print("contribute equally to distance calculations.")

---

## Part 4: K-Means Clustering

We use the elbow method and silhouette score to choose the optimal number of clusters, then assign each record to a cluster.

In [ ]:
# ─────────────────────────────────────────────────────
# ELBOW METHOD + SILHOUETTE SCORE
# ─────────────────────────────────────────────────────

K_range = range(2, 12)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(K_range, inertias, 'bo-', linewidth=2)
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method', fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(K_range, silhouettes, 'rs-', linewidth=2)
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Analysis', fontweight='bold')
ax2.grid(True, alpha=0.3)

best_k = list(K_range)[np.argmax(silhouettes)]
ax2.axvline(x=best_k, color='green', linestyle='--', label=f'Best k={best_k}')
ax2.legend()

plt.tight_layout()
plt.show()
print(f"Best k by silhouette score: {best_k}")

In [ ]:
# ─────────────────────────────────────────────────────
# FIT K-MEANS WITH CHOSEN k
# ─────────────────────────────────────────────────────

# Use at least 5 clusters for practical ground motion selection
n_clusters = max(best_k, 5)
km_final = KMeans(n_clusters=n_clusters, n_init=20, random_state=42)
df['KMeans_Cluster'] = km_final.fit_predict(X)

print(f"K-Means with k={n_clusters}")
print(f"\nCluster sizes:")
print(df['KMeans_Cluster'].value_counts().sort_index())
print(f"\nSilhouette score: {silhouette_score(X, df['KMeans_Cluster']):.4f}")

In [ ]:
# ─────────────────────────────────────────────────────
# VISUALIZE K-MEANS CLUSTERS
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PGA vs PGV colored by cluster
scatter = axes[0].scatter(df['PGA_g'], df['PGV_cm_s'], 
                          c=df['KMeans_Cluster'], cmap='tab10',
                          s=50, alpha=0.7, edgecolors='white', linewidth=0.5)
axes[0].set_xlabel('PGA (g)')
axes[0].set_ylabel('PGV (cm/s)')
axes[0].set_title('K-Means Clusters: PGA vs PGV', fontweight='bold')
plt.colorbar(scatter, ax=axes[0], label='Cluster')
axes[0].grid(True, alpha=0.3)

# PGA vs Duration colored by cluster
scatter2 = axes[1].scatter(df['PGA_g'], df['Duration_s'], 
                           c=df['KMeans_Cluster'], cmap='tab10',
                           s=50, alpha=0.7, edgecolors='white', linewidth=0.5)
axes[1].set_xlabel('PGA (g)')
axes[1].set_ylabel('Duration (s)')
axes[1].set_title('K-Means Clusters: PGA vs Duration', fontweight='bold')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Part 5: Hierarchical Clustering with Dendrogram

In [ ]:
# ─────────────────────────────────────────────────────
# HIERARCHICAL CLUSTERING DENDROGRAM
# ─────────────────────────────────────────────────────

Z = linkage(X, method='ward')

fig, ax = plt.subplots(figsize=(16, 6))
dendrogram(Z, truncate_mode='lastp', p=20,
           leaf_rotation=90, leaf_font_size=10,
           show_contracted=True, ax=ax)
ax.set_title('Hierarchical Clustering Dendrogram (Ward Linkage)', fontweight='bold')
ax.set_xlabel('Cluster / Sample Index')
ax.set_ylabel('Distance')
ax.axhline(y=15, color='red', linestyle='--', alpha=0.7, label='Cut threshold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────
# FIT AGGLOMERATIVE CLUSTERING
# ─────────────────────────────────────────────────────

agg = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
df['Hier_Cluster'] = agg.fit_predict(X)

print(f"Hierarchical clustering with {n_clusters} clusters")
print(f"\nCluster sizes:")
print(df['Hier_Cluster'].value_counts().sort_index())
print(f"\nSilhouette score: {silhouette_score(X, df['Hier_Cluster']):.4f}")

---

## Part 6: Select Representative Motions

For each cluster, we select the record closest to the cluster centroid. This gives us the most typical member of each group.

In [ ]:
# ─────────────────────────────────────────────────────
# SELECT REPRESENTATIVE MOTIONS (K-MEANS)
# ─────────────────────────────────────────────────────

def select_representatives(X, labels, df_records, method_name):
    """Select the record closest to centroid in each cluster."""
    reps = []
    unique_labels = sorted(set(labels))
    
    for c in unique_labels:
        mask = labels == c
        cluster_points = X[mask]
        cluster_indices = np.where(mask)[0]
        
        centroid = cluster_points.mean(axis=0).reshape(1, -1)
        dists = cdist(centroid, cluster_points, metric='euclidean')[0]
        best_idx = cluster_indices[np.argmin(dists)]
        
        rep = df_records.iloc[best_idx].copy()
        rep['Cluster'] = c
        rep['Dist_to_Centroid'] = round(dists.min(), 4)
        rep['Cluster_Size'] = mask.sum()
        reps.append(rep)
    
    result = pd.DataFrame(reps)
    print(f"\n{method_name}: Selected {len(result)} representative motions")
    return result

km_reps = select_representatives(X, df['KMeans_Cluster'].values, df, 'K-Means')
km_reps[['Record_ID', 'Source', 'PGA_g', 'PGV_cm_s', 'Duration_s', 
         'Cluster', 'Cluster_Size', 'Dist_to_Centroid']]

In [ ]:
# ─────────────────────────────────────────────────────
# SELECT REPRESENTATIVE MOTIONS (HIERARCHICAL)
# ─────────────────────────────────────────────────────

hier_reps = select_representatives(X, df['Hier_Cluster'].values, df, 'Hierarchical')
hier_reps[['Record_ID', 'Source', 'PGA_g', 'PGV_cm_s', 'Duration_s', 
           'Cluster', 'Cluster_Size', 'Dist_to_Centroid']]

---

## Part 7: Compare Selected Motions vs Full Library

In [ ]:
# ─────────────────────────────────────────────────────
# COMPARE SELECTED MOTIONS VS FULL LIBRARY
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# PGA vs PGV
axes[0].scatter(df['PGA_g'], df['PGV_cm_s'], c='lightgrey', s=30, 
                alpha=0.5, label='All records')
axes[0].scatter(km_reps['PGA_g'], km_reps['PGV_cm_s'], c='red', s=100, 
                marker='*', zorder=5, label='K-Means reps')
axes[0].scatter(hier_reps['PGA_g'], hier_reps['PGV_cm_s'], c='blue', s=80, 
                marker='D', zorder=5, label='Hier. reps', alpha=0.8)
axes[0].set_xlabel('PGA (g)')
axes[0].set_ylabel('PGV (cm/s)')
axes[0].set_title('PGA vs PGV', fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# PGA vs Duration
axes[1].scatter(df['PGA_g'], df['Duration_s'], c='lightgrey', s=30, alpha=0.5)
axes[1].scatter(km_reps['PGA_g'], km_reps['Duration_s'], c='red', s=100, marker='*', zorder=5)
axes[1].scatter(hier_reps['PGA_g'], hier_reps['Duration_s'], c='blue', s=80, marker='D', zorder=5)
axes[1].set_xlabel('PGA (g)')
axes[1].set_ylabel('Duration (s)')
axes[1].set_title('PGA vs Duration', fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Sa(0.2) vs Sa(1.0)
axes[2].scatter(df['Sa_02_g'], df['Sa_10_g'], c='lightgrey', s=30, alpha=0.5)
axes[2].scatter(km_reps['Sa_02_g'], km_reps['Sa_10_g'], c='red', s=100, marker='*', zorder=5)
axes[2].scatter(hier_reps['Sa_02_g'], hier_reps['Sa_10_g'], c='blue', s=80, marker='D', zorder=5)
axes[2].set_xlabel('Sa(0.2s) (g)')
axes[2].set_ylabel('Sa(1.0s) (g)')
axes[2].set_title('Sa(0.2s) vs Sa(1.0s)', fontweight='bold')
axes[2].grid(True, alpha=0.3)

fig.suptitle('Selected Representative Motions vs Full Library', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────
# COVERAGE STATISTICS
# ─────────────────────────────────────────────────────

print("Coverage of Selected Motions")
print("=" * 55)

for method, reps in [('K-Means', km_reps), ('Hierarchical', hier_reps)]:
    print(f"\n{method} Representatives:")
    print(f"  Records selected: {len(reps)} out of {len(df)}")
    
    for col in ['PGA_g', 'PGV_cm_s', 'Duration_s', 'Arias_m_s']:
        full_range = df[col].max() - df[col].min()
        sel_range = reps[col].max() - reps[col].min()
        coverage = (sel_range / full_range) * 100 if full_range > 0 else 0
        print(f"  {col:12s}: range coverage = {coverage:.1f}%")
    
    src_covered = reps['Source'].nunique()
    src_total = df['Source'].nunique()
    print(f"  Source types covered: {src_covered}/{src_total}")

---

## Part 8: Cluster Profiles and Summary

In [ ]:
# ─────────────────────────────────────────────────────
# CLUSTER PROFILES (K-MEANS)
# ─────────────────────────────────────────────────────

profile = df.groupby('KMeans_Cluster')[feature_cols].mean().round(4)
profile['Count'] = df.groupby('KMeans_Cluster').size()

print("K-Means Cluster Profiles (mean values):")
profile

In [ ]:
# ─────────────────────────────────────────────────────
# NORMALIZED BAR CHART OF CLUSTER PROFILES
# ─────────────────────────────────────────────────────

profile_norm = profile[feature_cols].copy()
for col in feature_cols:
    col_min, col_max = profile_norm[col].min(), profile_norm[col].max()
    if col_max > col_min:
        profile_norm[col] = (profile_norm[col] - col_min) / (col_max - col_min)

fig, ax = plt.subplots(figsize=(14, 5))
profile_norm.plot(kind='bar', ax=ax, width=0.8, edgecolor='white')
ax.set_xlabel('Cluster')
ax.set_ylabel('Normalized Value (0-1)')
ax.set_title('K-Means Cluster Profiles (Normalized Features)', fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Interpretation
print("\nCluster interpretation:")
for c in range(n_clusters):
    row = profile.loc[c]
    dominant_source = df[df['KMeans_Cluster'] == c]['Source'].mode().iloc[0]
    print(f"  Cluster {c}: {int(row['Count'])} records | "
          f"PGA={row['PGA_g']:.3f}g | Dur={row['Duration_s']:.1f}s | "
          f"Dominant: {dominant_source}")

---

## Key Takeaways

1. **Clustering groups similar ground motions** into families based on intensity measures (PGA, PGV, duration, spectral accelerations), reducing 100 records to a handful of representative motions.

2. **K-Means is fast and effective** for ground motion selection. The silhouette score and elbow method provide objective criteria for choosing the number of clusters.

3. **Hierarchical clustering with dendrograms** provides a visual tool for understanding how records relate and for choosing the cut level interactively.

4. **Representative selection via centroid proximity** ensures each selected motion is the most typical member of its cluster -- not an extreme outlier.

5. **Feature standardization is critical** because IMs have very different scales (PGA in g vs duration in seconds vs PGV in cm/s).

---

## Next Steps

- Day 094: Build an automated pipeline for geotechnical data analysis
- Extend this with conditional mean spectrum (CMS) matching
- Apply to real PEER NGA-West2 ground motion database

---

*Day 093 of 100 -- Clustering finds the signal in the seismic noise!*